# P13 - Recherche d'hyperparamètres

Ce notebook explore et compare les hyperparamètres optimaux pour chaque modèle.
Il complète l'entraînement automatique de `train.py` en offrant une visualisation fine des résultats MLflow.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import RandomizedSearchCV, train_test_split, cross_val_score
from sklearn.metrics import r2_score
import mlflow

if Path('/app/src').exists():
    sys.path.insert(0, '/app/src')
    MLFLOW_URI = 'http://mlflow:5000'
else:
    sys.path.insert(0, '../../src')
    MLFLOW_URI = 'http://localhost:5001'

from p13.db import read_sql
from p13.ml.features import ALL_TARGETS, FEATURE_COLUMNS

mlflow.set_tracking_uri(MLFLOW_URI)
print('MLflow tracking URI:', MLFLOW_URI)

## 1. Chargement des données

On utilise le dataset ML complet (toutes rentrees confondues).

In [ ]:
df = read_sql('SELECT * FROM ml_dataset_commune WHERE nb_eleves_maternelle IS NOT NULL')
df = df.dropna(subset=FEATURE_COLUMNS + ALL_TARGETS).copy()
print(f'{len(df)} lignes, {df.rentree.nunique()} rentrees, {df.code_insee.nunique()} communes')
X = df[FEATURE_COLUMNS].values
print('Features:', FEATURE_COLUMNS)

## 2. Grilles d'hyperparamètres

On définit des grilles larges pour `RandomizedSearchCV` (n_iter=30, cv=5).

In [ ]:
TARGET = 'nb_eleves_elementaire'
y = df[TARGET].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_grids = {
    'GradientBoosting': (
        GradientBoostingRegressor(random_state=42),
        {
            'n_estimators': [50, 100, 200, 300],
            'max_depth': [2, 3, 4, 5, 6],
            'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
            'subsample': [0.6, 0.8, 1.0],
            'min_samples_leaf': [1, 2, 4, 8],
        }
    ),
    'RandomForest': (
        RandomForestRegressor(random_state=42, n_jobs=-1),
        {
            'n_estimators': [50, 100, 200, 300],
            'max_depth': [4, 6, 8, 10, None],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', 0.5, 0.7],
        }
    ),
    'Ridge': (
        Ridge(),
        {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}
    ),
}
print('Modeles:', list(param_grids.keys()))

## 3. Recherche randomisée

`RandomizedSearchCV` avec 30 itérations et 5 folds. Les résultats sont loggués dans MLflow.

In [ ]:
results = []

for model_name, (estimator, param_dist) in param_grids.items():
    print(f'\nRecherche {model_name}...')
    search = RandomizedSearchCV(
        estimator,
        param_distributions=param_dist,
        n_iter=30,
        cv=5,
        scoring='r2',
        random_state=42,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
    )
    search.fit(X_train, y_train)
    best = search.best_estimator_
    r2_test = r2_score(y_test, best.predict(X_test))
    cv_scores = cross_val_score(best, X, y, cv=5, scoring='r2')

    with mlflow.start_run(run_name=f'hyperparam_{model_name}_{TARGET}'):
        mlflow.log_param('target', TARGET)
        mlflow.log_param('model', model_name)
        mlflow.log_params({f'best_{k}': str(v) for k, v in search.best_params_.items()})
        mlflow.log_metric('r2_test', r2_test)
        mlflow.log_metric('cv_r2_mean', float(cv_scores.mean()))
        mlflow.log_metric('cv_r2_std', float(cv_scores.std()))

    results.append({
        'model': model_name,
        'r2_test': round(r2_test, 4),
        'cv_r2_mean': round(cv_scores.mean(), 4),
        'cv_r2_std': round(cv_scores.std(), 4),
        'best_params': search.best_params_,
        'best_score_cv': round(search.best_score_, 4),
    })
    print(f'  Best CV R²={search.best_score_:.4f} | Test R²={r2_test:.4f} | Params={search.best_params_}')

df_results = pd.DataFrame([{k: v for k, v in r.items() if k != 'best_params'} for r in results])
df_results.sort_values('r2_test', ascending=False)

## 4. Comparaison visuelle

**R² test** et **R² CV** permettent de détecter le surapprentissage (R² test << R² CV).

In [ ]:
fig = px.bar(
    df_results.sort_values('cv_r2_mean'),
    x='cv_r2_mean', y='model',
    orientation='h',
    error_x='cv_r2_std',
    title=f'R² cross-validation moyen ± std — {TARGET}',
    labels={'cv_r2_mean': 'R² CV moyen', 'model': 'Modele'},
    color='cv_r2_mean',
    color_continuous_scale='Greens'
)
fig.show()

## 5. Meilleurs paramètres par modèle

In [ ]:
for r in sorted(results, key=lambda x: -x['r2_test']):
    print(f"\n=== {r['model']} (R2 test={r['r2_test']}, CV={r['cv_r2_mean']}±{r['cv_r2_std']}) ===")
    for k, v in r['best_params'].items():
        print(f'  {k}: {v}')

## 6. Courbes de validation (GradientBoosting)

On visualise l'impact de `n_estimators` sur le R² pour détecter le point d'overfitting.

In [ ]:
gb = GradientBoostingRegressor(random_state=42)

# Récupère les meilleurs params GB
best_gb = next(r['best_params'] for r in results if r['model'] == 'GradientBoosting')

n_range = [10, 25, 50, 100, 150, 200, 300]
val_scores = []
for n in n_range:
    params = {**best_gb, 'n_estimators': n}
    m = GradientBoostingRegressor(**params, random_state=42)
    m.fit(X_train, y_train)
    r2_tr = r2_score(y_train, m.predict(X_train))
    r2_te = r2_score(y_test, m.predict(X_test))
    val_scores.append({'n_estimators': n, 'R2_train': r2_tr, 'R2_test': r2_te})

df_val = pd.DataFrame(val_scores)
fig2 = px.line(
    df_val.melt(id_vars='n_estimators', var_name='split', value_name='R2'),
    x='n_estimators', y='R2', color='split',
    markers=True,
    title=f'Courbe de validation GradientBoosting — {TARGET}'
)
fig2.show()

## 7. Conclusions

- Comparer les R² test et CV : un écart important (> 0.05) indique du surapprentissage.
- Les meilleurs paramètres identifiés ici peuvent être reportés dans `train.py` → `PARAM_GRIDS` pour l'entraînement automatique.
- Relire les runs dans MLflow UI (http://localhost:5001) pour l'historique complet.
- La calibration de `_children_from_housing` (ratio 0.35 enfants/logement) reste à valider sur données locales.